# HDBSCAN_damage_shape.ipynb

Performs unsupervised clustering of damage shape polygons using HDBSCAN.

This notebook uses functions stored in damage_shape_tools.py.

After running this notebook you can:
- have a look at some of the views set up in the database

### Convert to python script using:
```
jupyter nbconvert --no-prompt --to script HDBSCAN_damage_shape.ipynb
```

## References
- [Gemini reference](https://share.gemini.google/OH0lJw2upRny)
- [Tuning DBSCAN parameters](https://share.gemini.google/vN3GhPCw2g3x)
- [Gemini reference for standard hdbscan module](https://share.google/aimode/EBb5xLhfQzgcLtEMP)

### Spatialite SQL to find polygons near left, right and top edges of image

```sql
UPDATE trees 
SET tree_touches_edge = 1 
FROM (
  SELECT 
    t.tree_id,
    - ST_MaxY(t.tree_poly) < 5 AS near_top,
    ST_MinX(t.tree_poly) < 5 AS near_left,
    i.image_width - ST_MaxX(t.tree_poly) < 5 AS near_right
  FROM trees t
  JOIN images i ON i.image_id = t.image_id
) AS sub
WHERE trees.tree_id = sub.tree_id  -- Crucial: links the update to the subquery
  AND (sub.near_top OR sub.near_left OR sub.near_right);
```

In [1]:
import tomllib
from contextlib import contextmanager
from datetime import datetime

import icecream
icecream.ic.configureOutput(includeContext=False) # silence warnings
from icecream import ic

from damage_shape_tools import run_train_damage_model_shape_pipeline, run_damage_shape_classifier_pipeline

# FUNCTIONS

In [2]:
def timestamp():
    """ 
    use for ic formatting 
    """
    return f"{datetime.now().isoformat(sep='T', timespec='seconds')} "

#########################################################################


In [3]:
@contextmanager
def ic_red():
    """ 
    displays ic message in red while within the context 
    """
    RED, RESET = "\033[31m", "\033[0m"
    ic.configureOutput(outputFunction=lambda s: print(f"{RED}{s}{RESET}"))
    try:
        yield
    finally:
        # Automatically revert back to normal
        ic.configureOutput(outputFunction=print)

# # Usage Example:
# ic("Normal color")
# with ic_red():
#     ic("Temporary red message 1")
#     ic("Temporary red message 2")
# ic("Normal color again")

#########################################################################

In [4]:
def run_task(task: str):
    """  
    Task can be 'TRAIN DAMAGE SHAPE MODEL' or 'CLASSIFY DAMAGE SHAPES'
    """
    ic.configureOutput(prefix=timestamp, includeContext=True)

    ic('starting')

    ic('getting parameters from config.toml')
    with open("config.toml", "rb") as f:
        config = tomllib.load(f)  
    ic(config['database'])
    ic(config['damage'])  

    if task =='TRAIN DAMAGE SHAPE MODEL':
        ic(task)
        run_train_damage_model_shape_pipeline(
            db_path=config['database']['db_path'], 
            db_backup_dir=config['database']['db_backup_dir'], 
            model_path=config['damage']['model_path'], 
            images_per_cluster=config['damage']['images_per_cluster'], 
            gallery_dir=config['damage']['gallery_dir'], 
            min_prob=config['damage']['min_prob']) 
        
        WARNING = f'IMPORTANT: Please have look at images in the damage cluster gallery ({config['damage']['gallery_dir']}) and update {config['damage']['csv_path']} before proceeding with the CLASSIFY TREE SHAPES task'
        with ic_red():
            ic(WARNING) 

    if task == 'CLASSIFY DAMAGE SHAPES':
        ic(task)
        run_damage_shape_classifier_pipeline(
            db_path=config['database']['db_path'], 
            csv_path=config['damage']['csv_path']
            )
            
    ic('finished');

# MAIN

In [5]:
run_task('TRAIN DAMAGE SHAPE MODEL')
# run_task('CLASSIFY DAMAGE SHAPES')

2026-09-12T16:22:38 126613065.py:7 in run_task()- 'starting'
2026-09-12T16:22:38 126613065.py:9 in run_task()
                    'getting parameters from config.toml': 'getting parameters from config.toml'
2026-09-12T16:22:38 126613065.py:12 in run_task()
                    config['database']: {'db_backup_dir': 'db_backups',
                                         'db_path': 'Efate2025B_4k.db',
                                         'delete_db': True}
2026-09-12T16:22:38 126613065.py:13 in run_task()
                    config['damage']: {'csv_path': 'damage_cluster2class.csv',
                                       'gallery_dir': 'damage_cluster_gallery_1',
                                       'images_per_cluster': 35,
                                       'min_prob': 0.2,
                                       'minpixels': 50,
                                       'model_path': 'hdbscan_damage_pipeline.joblib',
                                       'order': 14}
2026-09-12T1

OperationalError: no such column: tree_touches_edge